### Imports

In [2]:
import pandas as pd
import pickle
import re

from scipy.sparse import hstack

from sklearn.metrics import (accuracy_score, classification_report)

### Load Saved Files

In [3]:
# Load Best Model
with open("../models/best_intent_classifier.pkl", "rb") as f:
    best_model = pickle.load(f)

# Load Word Vectorizer
with open("../models/word_vectorizer.pkl", "rb") as f:
    word_vectorizer = pickle.load(f)

# Load Character Vectorizer
with open("../models/char_vectorizer.pkl", "rb") as f:
    char_vectorizer = pickle.load(f)

# Load Label Encoder
with open("../models/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

print("All models and vectorizers loaded successfully!")

All models and vectorizers loaded successfully!


### Preprocessing Function

In [4]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### Prediction Function

In [5]:
def predict_intent(query):

    # Preprocess
    cleaned_query = preprocess_text(query)

    # Word Features
    word_features = word_vectorizer.transform(
        [cleaned_query]
    )

    # Character Features
    char_features = char_vectorizer.transform(
        [cleaned_query]
    )

    # Combine Features
    combined_features = hstack([
        word_features,
        char_features
    ])

    # Predict
    prediction = best_model.predict(
        combined_features
    )

    # Decode Label
    predicted_label = label_encoder.inverse_transform(
        prediction
    )[0]

    return predicted_label

### Load Verification Dataset

In [6]:
verification_df = pd.read_csv(
    "../data/verification_data/banking_knowledge_base_1000.csv"
)

verification_df.head()

,Section,Question,Answer
0,Basic Accounts,What is a Savings Account?,A savings account lets you keep your money saf...
1,Basic Accounts,What is a Current Account?,A current account is mainly for businesses. It...
2,Basic Accounts,What is a Fixed Deposit?,A fixed deposit lets you lock in a sum of mone...
3,Basic Accounts,What does KYC stand for?,KYC stands for Know Your Customer. It's how ba...
4,Basic Accounts,What is a cheque?,A cheque is a written instruction to your bank...


### Predict Intents

In [7]:
verification_df["Predicted_Intent"] = verification_df[
    "Question"
].apply(predict_intent)

print(
    verification_df[
        ["Question", "Predicted_Intent"]
    ].sample(5)
)

                                              Question  \
965                What is a closed-ended mutual fund?   
616         What is the Senior Citizen Savings Scheme?   
915                         What is loan amortisation?   
938  How do I get a no-objection certificate from a...   
984  What is the difference between bank credit and...   

             Predicted_Intent  
965         Financial Markets  
616  Insurance & Govt Schemes  
915          Loans & Interest  
938          Customer Service  
984       Banking Terminology  


### Save Prediction File

In [8]:
verification_df.to_csv(
    "../data/verification_data/predicted_output.csv",
    index=False
)

print("Prediction file saved successfully!")

Prediction file saved successfully!


### Calculate Accuracy

In [9]:
accuracy = accuracy_score(
    verification_df["Section"],
    verification_df["Predicted_Intent"]
)

print(
    f"Verification Accuracy: "
    f"{accuracy:.4f}"
)

Verification Accuracy: 0.9700


### Classification Report

In [10]:
print("\nClassification Report:\n")

print(
    classification_report(
        verification_df["Section"],
        verification_df["Predicted_Intent"]
    )
)


Classification Report:

                          precision    recall  f1-score   support

     Banking Terminology       0.94      0.94      0.94        86
          Basic Accounts       0.98      0.98      0.98       121
        Customer Service       0.99      1.00      0.99        91
         Digital Banking       0.98      0.95      0.96       116
       Financial Markets       0.97      0.95      0.96        92
         General Finance       0.92      0.97      0.94       106
Insurance & Govt Schemes       0.99      0.98      0.98        90
   International Banking       0.99      0.97      0.98        79
        Loans & Interest       0.98      0.99      0.99       110
   Operations & Security       0.96      0.97      0.97       109

                accuracy                           0.97      1000
               macro avg       0.97      0.97      0.97      1000
            weighted avg       0.97      0.97      0.97      1000



### Prediction Status

In [11]:
verification_df["Prediction_Status"] = (
    verification_df["Section"] == verification_df["Predicted_Intent"]
)

verification_df.sample(5)

,Section,Question,Answer,Predicted_Intent,Prediction_Status
182,Digital Banking,What is the National Financial Switch?,NFS is the ATM network run by NPCI that connec...,Digital Banking,True
818,Financial Markets,What is the yield curve?,The yield curve plots interest rates of bonds ...,Financial Markets,True
194,Digital Banking,What is a SWIFT code?,SWIFT code is a unique code identifying your b...,Operations & Security,False
168,Digital Banking,What is a credit card billing cycle?,A billing cycle is the monthly period (usually...,Digital Banking,True
394,Operations & Security,What is an alert transaction?,An alert transaction is any transaction that t...,Operations & Security,True


### Correct Predictions Count

In [12]:
correct_predictions = (
    verification_df["Prediction_Status"].sum()
)

total_predictions = len(verification_df)

print("Correct Predictions:", correct_predictions)
print("Total Predictions:", total_predictions)

Correct Predictions: 970
Total Predictions: 1000


### Accuracy Percentage

In [13]:
accuracy_percent = accuracy * 100

print(
    f"Model Accuracy on Verification File: "
    f"{accuracy_percent:.2f}%"
)

Model Accuracy on Verification File: 97.00%


# Key Insights & Observations

## Verification Dataset Evaluation

A separate verification dataset was used for external validation.

### Observation
- The trained pipeline successfully predicted intents for unseen banking queries.
- Real-world testing improved confidence in model robustness.

### Evaluation Metrics
The notebook evaluated:
- training accuracy
- testing accuracy
- cross-validation accuracy
- overfitting gap

This provided a comprehensive performance assessment.